
 Create a dataframe as below
```
  +---+--------+---+------+
  | id|    name|age|salary|
  +---+--------+---+------+
  |100|Prashant| 45| 45000|
  |101|   Tarun| 36| 33000|
  |102|   David| 48| 28000|
  +---+--------+---+------+
```

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataTransformation").getOrCreate()

schema = "id int, name string, age short, salary double"

data_list = [(100, 'Prashant', 45, 45000.0),
             (101, 'Tarun', 36, 33000.0),
             (102, 'David', 48, 28000.0)]

sample_df = spark.createDataFrame(data_list, schema)

sample_df.show()

+---+--------+---+-------+
| id|    name|age| salary|
+---+--------+---+-------+
|100|Prashant| 45|45000.0|
|101|   Tarun| 36|33000.0|
|102|   David| 48|28000.0|
+---+--------+---+-------+



 Add following columns to your Dataframe
1. increment: 10% of the salary up to 3000 maximum increment
2. revised_salary: salary + increment

In [ ]:
from pyspark.sql.functions import expr 

salary_df = (
    sample_df.withColumns({
            "increment" : expr(" case when salary > 30000 then 3000 else salary * 0.10 end"),
            "revised_salary" : expr("salary + increment")   
    })
)

salary_df.show()

+---+--------+---+-------+---------+--------------+
| id|    name|age| salary|increment|revised_salary|
+---+--------+---+-------+---------+--------------+
|100|Prashant| 45|45000.0|   3000.0|       48000.0|
|101|   Tarun| 36|33000.0|   3000.0|       36000.0|
|102|   David| 48|28000.0|   2800.0|       30800.0|
+---+--------+---+-------+---------+--------------+



Add following columns to your Dataframe
* increment: 10% of the salary up to 3000 maximum increment

Replace the following column in your dataframe
* salary: current salary + increment

Meaning instead of new revised_salary - the same salary column in the DF should have new salaries

Approach -1 - using the incremental logic in revised_salary column

In [5]:
from pyspark.sql.functions import expr 

salary_df = (
    sample_df.withColumns({
            "increment" : expr(" case when salary > 30000 then 3000 else salary * 0.10 end"),
            "revised_salary" : expr("salary + case when salary > 30000 then 3000 else salary * 0.10 end")   
    })
)

salary_df.show()

+---+--------+---+-------+---------+--------------+
| id|    name|age| salary|increment|revised_salary|
+---+--------+---+-------+---------+--------------+
|100|Prashant| 45|45000.0|   3000.0|       48000.0|
|101|   Tarun| 36|33000.0|   3000.0|       36000.0|
|102|   David| 48|28000.0|   2800.0|       30800.0|
+---+--------+---+-------+---------+--------------+



Approach -2 - When transforming already available column.it is little slow in processing when using chain of withColumns (Try avoiding).

In [7]:
from pyspark.sql.functions import expr 

salary_df = (
    sample_df.withColumn("increment" , expr(" case when salary > 30000 then 3000 else salary * 0.10 end"))
            .withColumn("revised_salary", expr("salary + increment"))   
    
)

salary_df.show()

+---+--------+---+-------+---------+--------------+
| id|    name|age| salary|increment|revised_salary|
+---+--------+---+-------+---------+--------------+
|100|Prashant| 45|45000.0|   3000.0|       48000.0|
|101|   Tarun| 36|33000.0|   3000.0|       36000.0|
|102|   David| 48|28000.0|   2800.0|       30800.0|
+---+--------+---+-------+---------+--------------+



Add a batch number (uuid) column to your dataframe

In [10]:
import uuid

from pyspark.sql.functions import lit

batch_id = str(uuid.uuid4())

salary_batch_df = sample_df.withColumn("batch_id", lit(batch_id))

salary_batch_df.show()

+---+--------+---+-------+--------------------+
| id|    name|age| salary|            batch_id|
+---+--------+---+-------+--------------------+
|100|Prashant| 45|45000.0|ccccb620-a288-461...|
|101|   Tarun| 36|33000.0|ccccb620-a288-461...|
|102|   David| 48|28000.0|ccccb620-a288-461...|
+---+--------+---+-------+--------------------+



Rename the dataframe colums as listed below
1. increment - annual_increment
2. salary - incremented_salary

In [11]:
new_salary_df = (
    salary_df.withColumnsRenamed({
        "increment" : "annual_increment",
        "salary" : "incremented_salary"
    })
)

new_salary_df.show()

+---+--------+---+------------------+----------------+--------------+
| id|    name|age|incremented_salary|annual_increment|revised_salary|
+---+--------+---+------------------+----------------+--------------+
|100|Prashant| 45|           45000.0|          3000.0|       48000.0|
|101|   Tarun| 36|           33000.0|          3000.0|       36000.0|
|102|   David| 48|           28000.0|          2800.0|       30800.0|
+---+--------+---+------------------+----------------+--------------+



Remove the following colums from your dataframe
1. age
2. annual_increment

In [12]:
small_salary_df = (
    new_salary_df.drop("age","annual_increment")
)

small_salary_df.show()

+---+--------+------------------+--------------+
| id|    name|incremented_salary|revised_salary|
+---+--------+------------------+--------------+
|100|Prashant|           45000.0|       48000.0|
|101|   Tarun|           33000.0|       36000.0|
|102|   David|           28000.0|       30800.0|
+---+--------+------------------+--------------+

